# 03 — Exploratory data analysis and statistics

This notebook is a thin presentation layer over the shared `src/` analysis and `docs/results.json`. It describes the supplied file, compares the four documented cleaning views, displays the seven handbook figures, and reviews the pre-specified H1–H5 results. It does not fit a predictive model or make clinical claims.

Run `python -m src.run_analysis` from the repository root before opening this notebook if the generated results or figures are missing. The supplied file's checksum differs from the project contract; this notebook uses the explicitly approved current-file override and continues to label outcomes as the recorded label.

In [ ]:
from pathlib import Path
import json
import sys

candidates = [Path.cwd(), *Path.cwd().parents]
ROOT = next(
    (candidate for candidate in candidates if (candidate / "docs" / "results.json").is_file()),
    None,
)
if ROOT is None:
    raise FileNotFoundError("Start Jupyter from the repository root or one of its subfolders.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import pandas as pd
from IPython.display import Image, display

from src.clean import make_views
from src.config import DOCS, FIG, SEED
from src.data_io import load_raw

results_path = DOCS / "results.json"
if not results_path.is_file():
    raise FileNotFoundError(f"Run `python -m src.run_analysis` first: {results_path}")
results = json.loads(results_path.read_text(encoding="utf-8"))
print(f"Project root: {ROOT}")
print(f"Random seed: {SEED}")
print(f"Registered figures: {len(results['figure_files'])}")

## 1. Describe before testing

Load through the project data interface and confirm that the working copy agrees with the generated result manifest. The identifier is excluded from descriptive feature summaries because it is not an analysis feature.

In [ ]:
df = load_raw(verify=False)
views = make_views(df)
dataset = results["dataset"]
assert df.shape == (dataset["n_raw"], dataset["n_columns"])
assert int(df.isna().sum().sum()) == dataset["null_count"]
assert df["patient_id"].is_unique == dataset["patient_id_unique"]

descriptive = df.drop(columns="patient_id").describe().T
descriptive["skew"] = df.drop(columns="patient_id").skew(numeric_only=True)
descriptive["kurtosis"] = df.drop(columns="patient_id").kurtosis(numeric_only=True)
print(f"Shape: {df.shape}; null cells: {dataset['null_count']}")
print(f"Recorded-label code 1 share: {results['target']['positive_rate']:.3%}")
display(descriptive.round(3))

## 2. Compare the four analysis views

The views are sensitivity policies, not claims that one version is ground truth. The D view excludes the leading ordered block after deduplication.

In [ ]:
view_summary = pd.DataFrame(
    [
        {
            "view": name,
            "rows": len(frame),
            "recorded_label_1_share": frame["LUNG_CANCER"].mean(),
        }
        for name, frame in views.items()
    ]
)
for row in view_summary.itertuples(index=False):
    expected_view = results["views"][row.view]
    assert row.rows == expected_view["n"]
    assert abs(row.recorded_label_1_share - expected_view["positive_rate"]) < 1e-12
display(view_summary)

## 3. Seven core figures

Each figure answers a handbook question. Captions and takeaways come from the generated figure manifest; images are displayed directly from `reports/figures/`.

In [ ]:
core_figures = [
    "f1_order.png",
    "v6_yes_share_by_block.png",
    "fig3_correlation.png",
    "v7_age_profiles.png",
    "fig5_views.png",
    "fig6_cramers_v.png",
    "fig7_smoking_forest.png",
]
registered = set(results["figure_files"])
for filename in core_figures:
    path = FIG / filename
    if filename not in registered or not path.is_file() or path.stat().st_size == 0:
        raise FileNotFoundError(f"Registered core figure is unavailable: {path}")
    details = results["figures"][filename]
    display(pd.DataFrame([[details["question"], details["takeaway"]]], columns=["Question", "Takeaway"]))
    display(Image(filename=str(path)))

## 4. Pre-specified hypotheses H1–H5

Effect sizes and confidence intervals accompany test statistics and adjusted q-values. H1–H4 are reported across the applicable views; H5 is the raw-file leading-block integrity comparison and is not repeated for view D.

In [ ]:
hypothesis_results = pd.DataFrame(results["hypothesis_results"])
assert set(hypothesis_results["hypothesis"]) == {"H1", "H2", "H3", "H4", "H5"}
assert hypothesis_results[["effect_size", "ci_low", "ci_high", "p", "q_BH"]].notna().all().all()
display(
    hypothesis_results.sort_values(["hypothesis", "view", "feature"], na_position="last")
    .reset_index(drop=True)
    .round(6)
)

print(
    "Interpretation: report the effect size and interval, distinguish nominal p-values "
    "from family-adjusted q-values, and limit conclusions to associations in this file."
)